# 01 — Raster inventory & native characterization

**Scope:** data ingestion + a table of each input dataset's **native** characteristics,
so we can see what alignment work is needed later. **Characterization only** — nothing
here reprojects, resamples, clips, or aligns. Reads are **metadata-only** (no full
pixel arrays); optional sampled stats use decimated overviews and are off by default.

**Kernel:** select **`Python (y2y-geo)`** (the project `.venv`, Python 3.12).

**Config-driven:** dataset paths, citations, and the proposed `resampling` method live
in **`config.py`** (shared with `02_preprocess_align.ipynb`) — add a dataset by adding an
entry there; the logic here doesn't change.

**Later-stage target grid:** **ESRI:102008 @ 1 km** (first iteration; see `config.py` /
CLAUDE.md). Not applied here — this notebook only records native properties.

Pipeline: discover rasters per dataset → characterize the representative raster → flag
full-corridor coverage → assemble a DataFrame shown inline.

In [1]:
import importlib

import rasterio
import pyproj
import pandas as pd
import geopandas as gpd

import config
importlib.reload(config)  # pick up edits to config.py when re-running this cell
from config import DATASETS, CORRIDOR_REF, PROJECT_DIR, find_rasters, pick_representative

print("rasterio", rasterio.__version__, "| geopandas", gpd.__version__, "| pyproj", pyproj.__version__)
print("datasets in config:", len(DATASETS))

rasterio 1.5.0 | geopandas 1.1.3 | pyproj 3.7.2
datasets in config: 9


In [2]:
import math

# ---- Coverage reference --------------------------------------------------
# Load the corridor polygon once. The coverage flag reprojects the actual
# corridor GEOMETRY into each raster's CRS (vertex transform via geopandas),
# then tests bbox containment. We must NOT transform only the corridor's
# bounding-box corners: Y2Y is a long diagonal region, so its axis-aligned bbox
# has large empty corners that bulge well beyond the real extent when reprojected
# -- that made fully-covering rasters (e.g. gHM, carbon) read as False.
CORRIDOR = gpd.read_file(CORRIDOR_REF)


def covers_corridor(raster_bounds, raster_crs):
    """True if the raster's native bbox contains the corridor's full extent.
    Returns None when the raster has no CRS (can't be compared without one)."""
    if raster_crs is None:
        return None
    target = pyproj.CRS.from_user_input(raster_crs)
    cminx, cminy, cmaxx, cmaxy = CORRIDOR.to_crs(target).total_bounds
    left, bottom, right, top = raster_bounds
    return (left <= cminx) and (bottom <= cminy) and (right >= cmaxx) and (top >= cmaxy)


# ---- CRS + characterization ---------------------------------------------
METRES_PER_DEGREE = 111_320  # approx mean metres per degree of latitude


def approx_res_metres(units, res_x, res_y, lat_center):
    """Approximate ground resolution in metres for geographic (degree) rasters.
    Latitude term is ~constant; longitude term shrinks by cos(latitude). Returns
    (x_m, y_m), or (None, None) for projected rasters (res already in metres)."""
    if units is None or "degree" not in units.lower():
        return (None, None)
    y_m = abs(res_y) * METRES_PER_DEGREE
    x_m = abs(res_x) * METRES_PER_DEGREE * math.cos(math.radians(lat_center))
    return (round(x_m), round(y_m))


def crs_descriptors(crs):
    """Return (code, name, units) for a rasterio CRS. code may be None if the CRS
    has no registered EPSG/ESRI authority code (full WKT only)."""
    if crs is None:
        return (None, None, None)
    cc = pyproj.CRS.from_user_input(crs)
    auth = cc.to_authority()  # e.g. ('EPSG', '4326') or ('ESRI', '102008')
    code = f"{auth[0]}:{auth[1]}" if auth else None
    unit = cc.axis_info[0].unit_name if cc.axis_info else None
    return (code, cc.name, unit)


def characterize(name, cfg):
    rasters = find_rasters(cfg)
    rep = pick_representative(cfg, rasters)
    total_size = sum(p.stat().st_size for p in rasters)
    with rasterio.open(rep) as src:  # metadata only -- no pixel reads
        code, crs_name, units = crs_descriptors(src.crs)
        b = src.bounds
        lat_center = (b.bottom + b.top) / 2  # only used when CRS is in degrees
        approx_x_m, approx_y_m = approx_res_metres(units, src.res[0], src.res[1], lat_center)
        return {
            "dataset": name,
            "representative_file": str(rep.relative_to(PROJECT_DIR)),
            "n_rasters": len(rasters),
            "driver": src.driver,
            "crs_code": code,
            "crs_name": crs_name,
            "crs_units": units,
            "res_x": src.res[0],
            "res_y": src.res[1],
            "approx_res_x_m": approx_x_m,  # ~metres for degree CRS; None if already metres
            "approx_res_y_m": approx_y_m,
            "resampling": cfg.get("resampling"),  # proposed method for 02 (from config)
            "width": src.width,
            "height": src.height,
            "bounds_left": b.left,
            "bounds_bottom": b.bottom,
            "bounds_right": b.right,
            "bounds_top": b.top,
            "band_count": src.count,
            "dtype": src.dtypes[0],
            "nodata": src.nodata,
            "file_size_mb": round(rep.stat().st_size / 1e6, 1),
            "dataset_total_mb": round(total_size / 1e6, 1),
            "covers_corridor": covers_corridor(
                (b.left, b.bottom, b.right, b.top), src.crs
            ),
            "citation": cfg.get("citation"),
        }

In [3]:
# Build the inventory: one row per dataset entry.
rows = [characterize(name, cfg) for name, cfg in DATASETS.items()]
inventory = pd.DataFrame(rows)
print(f"Characterized {len(inventory)} datasets.")

Characterized 9 datasets.


In [4]:
# Display the inventory inline (final output of this session -- no file export).
# The table is wide (~24 columns); show every column so nothing (e.g. `resampling`)
# is hidden behind pandas' middle-column truncation.
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
inventory

,dataset,representative_file,n_rasters,driver,crs_code,crs_name,crs_units,res_x,res_y,approx_res_x_m,approx_res_y_m,resampling,width,height,bounds_left,bounds_bottom,bounds_right,bounds_top,band_count,dtype,nodata,file_size_mb,dataset_total_mb,covers_corridor,citation
0,human_modification,input_data/human_modification/HM_Y2Y_2024_90_6...,4,GTiff,EPSG:4326,WGS 84,degree,0.000808,0.000808,52.0,90.0,average,32768,32768,-1.421411e+02,4.134020e+01,-1.156488e+02,6.783260e+01,1,float32,NaN,551.8,844.9,False,"Theobald et al. 2024, gHM v3 human modification"
1,transboundary_connectivity,input_data/transboundary_connectivity/Raw_Curr...,3,GTiff,EPSG:3347,unknown,metre,300.000000,300.000000,NaN,NaN,average,23812,24675,1.871836e+06,-1.601261e+06,9.015436e+06,5.801239e+06,1,float32,NaN,1083.4,1183.7,True,"Pither et al., transboundary omnidirectional c..."
2,climate_corridors,input_data/climate_corridors/centrality/curren...,1,GTiff,NaN,unnamed,metre,5000.000000,5000.000000,NaN,NaN,bilinear,1546,1616,-4.379400e+06,-3.697200e+06,3.350600e+06,4.382800e+06,1,float32,-3.400000e+38,4.2,4.2,True,"Carroll et al. 2018, current-flow centrality"
3,climate_type_macrorefugia,input_data/climate_type_macrorefugia/ensemble_...,6,GTiff,NaN,unknown,metre,1000.000000,1000.000000,NaN,NaN,bilinear,7505,7588,-4.169000e+06,-3.313000e+06,3.336000e+06,4.275000e+06,1,float32,-3.400000e+38,98.2,600.4,True,"Carroll 2023 (AdaptWest), backward climatic ve..."
4,irrecoverable_carbon_biomass,input_data/irrecoverable_carbon/irrecoverable_...,1,GTiff,EPSG:4326,WGS 84,degree,0.000889,0.000889,57.0,99.0,average,36828,28767,-1.410356e+02,4.175111e+01,-1.082996e+02,6.732178e+01,1,float64,-inf,880.7,880.7,True,"Berman/McDowell, irrecoverable carbon (biomass)"
5,irrecoverable_carbon_m_soc,input_data/irrecoverable_carbon/irrecoverable_...,1,GTiff,EPSG:4326,WGS 84,degree,0.002246,0.002246,145.0,250.0,average,14578,11387,-1.410378e+02,4.174919e+01,-1.082987e+02,6.732200e+01,1,float64,-inf,159.8,159.8,True,"Berman/McDowell, irrecoverable carbon (mineral..."
6,iucn_efg,input_data/iucn_efg/all-maps-raster-geotiff/F1...,109,GTiff,EPSG:4326,WGS 84,degree,0.008333,0.008333,928.0,928.0,nearest,43200,21600,-1.800000e+02,-9.000000e+01,1.800000e+02,9.000000e+01,1,uint8,0.000000e+00,7.4,282.2,True,"IUCN GET Ecosystem Functional Groups, Level 3"
7,aoh_richness_mammals,input_data/aoh_richness/Richness_mammals/Mamma...,1,GTiff,EPSG:4326,WGS 84,degree,0.000992,0.000992,109.0,110.0,average,362880,141120,-1.800005e+02,-5.999950e+01,1.799995e+02,8.000050e+01,1,int16,-3.276800e+04,3104.0,3104.0,True,"Lumbierres et al., AOH species richness (mamma..."
8,aoh_richness_birds,input_data/aoh_richness/Richness_birds/Birds_R...,1,GTiff,EPSG:4326,WGS 84,degree,0.000992,0.000992,109.0,110.0,average,362880,141120,-1.800005e+02,-5.999950e+01,1.799995e+02,8.000050e+01,1,int16,-3.276800e+04,4368.8,4368.8,True,"Lumbierres et al., AOH species richness (birds..."


## Optional: sampled value stats

Off by default. Set `COMPUTE_STATS = True` to compute min/max/mean from a **decimated**
read (a small downsampled overview) of each representative raster — this never loads a
full pixel array. Useful for a quick sense of value ranges; skip it for pure
metadata characterization.

In [6]:
COMPUTE_STATS = True  # leave False -- reads decimated overviews, never full arrays


def sampled_stats(path, max_dim=512):
    """Min/max/mean from a downsampled read of band 1 (never the full array)."""
    with rasterio.open(path) as src:
        scale = max(src.width, src.height) / max_dim
        out_w = max(1, int(src.width / scale))
        out_h = max(1, int(src.height / scale))
        arr = src.read(1, out_shape=(out_h, out_w), masked=True)
    return {"min": float(arr.min()), "max": float(arr.max()), "mean": float(arr.mean())}


if COMPUTE_STATS:
    _stats = {
        name: sampled_stats(pick_representative(cfg, find_rasters(cfg)))
        for name, cfg in DATASETS.items()
    }
    display(pd.DataFrame(_stats).T)

,min,max,mean
human_modification,NaN,NaN,NaN
transboundary_connectivity,0.000000,78.735329,1.724982
climate_corridors,0.000000,172.858887,5.276224
climate_type_macrorefugia,0.014946,35.119434,5.675104
irrecoverable_carbon_biomass,0.000000,180.800461,19.585907
irrecoverable_carbon_m_soc,0.000000,2164.250690,73.950733
iucn_efg,1.000000,2.000000,1.744156
aoh_richness_mammals,1.000000,190.000000,38.769442
aoh_richness_birds,1.000000,506.000000,109.654864
